### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

# MobileNet for Rain Drop Classification

This notebook repurposes the Raindrop Clarity image collection for binary classification of clear versus raindrop images. It adapts an ImageNet-pretrained MobileNetV3-small classifier with transfer learning, evaluates the resulting model on held-out data, exports representative tensors, converts the model for embedded deployment, and compares inference timing across supported targets.

## Notebook Structure

1. **Import libraries and helper functions**
2. **Download and restructure the Raindrop Clarity data**
3. **Select classes, create splits, and visualize inputs**
4. **Adapt the ImageNet-pretrained MobileNetV3-small model**
5. **Train the classification layer with transfer learning**
6. **Generate representative input and output data**
7. **Evaluate on samples and held-out test data**
8. **Export the model and test data**
9. **Convert the model for deployment**
10. **Display inference timing**
11. **Compare performance across hardware platforms**
12. **References and Generated Artifacts**

## 1. Import libraries and helper functions

Import PyTorch, torchvision, repository helpers, and the conversion/benchmarking utilities used throughout the workflow.

In [ ]:
import os
import sys

from torchsummary import summary
import modelling_helper as mh

import torch.nn as nn
import torch.optim as optim

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from _CentralScripts.python_flask_client import CallTools
import _CentralScripts.helper_functions as cs
import _CentralScripts.mobilenet_helper as mo

## 2. Download and restructure the Raindrop Clarity data

The notebook downloads and restructures the [Raindrop Clarity dataset](https://github.com/jinyeying/RaindropClarity) through `fetch_sort_data()`. Raindrop Clarity was created for image-based raindrop removal, but this example repurposes its images for classification. Only the clear and raindrop image categories are retained. The helper creates train, validation, and test folders; the validation and test images are used for model-quality checks and are not used to update the classifier.

The source dataset is a domain-specific image collection, so classification results may depend on its camera conditions, weather patterns, and class balance. The data are suitable for demonstrating transfer learning and embedded conversion, but they do not establish robustness across all vehicles, cameras, or weather conditions.

In [ ]:
batch_size = 16
path = mh.fetch_sort_data()
dataloader, class_names, dataset_sizes = mo.get_dataloader(
    os.path.join(path, "train"), batch_size, mode="train"
)
_, input_size = mh.get_input_dataloader(dataloader, is_plot=True)

## 3. Select classes, create splits, and visualize inputs

Build the train, validation, and test loaders, confirm the clear and raindrop class mapping, and visualize representative samples before adapting the model.

## 4. Adapt the ImageNet-pretrained MobileNetV3-small model

MobileNetV3-small is initialized with ImageNet1K pretrained weights [3]. Its final classification layer is replaced with a two-class head for clear and raindrop images. The pretrained feature extractor supplies general visual representations while the new head is adapted to this notebook's weather-classification task.

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

model_original = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
model = mh.adapt_model(model_original, number_classes=2)
summary(model, input_size, device="cpu")

## 5. Train the classification layer with transfer learning

Only the replacement classification layer is optimized with cross-entropy loss and SGD. This short training run demonstrates the workflow; increase the epoch count and record the configuration for a meaningful accuracy study.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.classifier[-1].parameters(), lr=0.01, momentum=0.9)
mh.train_model(model, dataloader, dataset_sizes, criterion, optimizer, num_epochs=1)

## 6. Generate representative input and output data

A representative transformed image tensor and the model output are captured for conversion validation and later embedded execution checks. These tensors define the exported model's input/output contract.

In [ ]:
input_target, _ = mh.get_input_dataloader(dataloader, is_plot=True)
output_target = cs.get_predictions("torch", model, input_target)

## 7. Evaluate on samples and held-out test data

The notebook first plots predictions for a few test samples, then evaluates the trained classifier over the held-out test split. These results assess classification behavior separately from the hardware timing and platform comparison performed later.

In [ ]:
classes = mh.get_rain_classes()
mo.calculate_predictions_and_plot(model, os.path.join(path, "test"), classes)

### Held-out test-set evaluation

Testing the trained model on the held-out test data set provides a separate quality check after the sample-level prediction visualization.

In [ ]:
dataloader, class_names, dataset_sizes = mo.get_dataloader(
    os.path.join(path, "test"), batch_size, mode="test"
)
mh.test_model(model, dataloader, dataset_sizes, criterion)

## 8. Export the model and test data

Save the trained PyTorch model as an ONNX artifact together with representative input and expected output data. The generated files are consumed by the conversion service.

In [ ]:
model_name = "mobilenet_rain_drop"
cs.save_all(model_name, input_target, output_target, model, origin="torch")

## 9. Convert the model for deployment

Start the Docker conversion service and submit the ONNX model with representative test data. Generated source, binaries, logs, timing, and profile results are written below `out/<model_name>/test_<model_name>/<target>/`. The deployment targets are AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.

In [ ]:
cs.ensure_docker_container()

In [ ]:
model_folder, onnx_model_file = cs.get_output_paths(model_name)

target = "tc4dx"  # Available targets: "tc3xx", "tc4dx"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    profile=True,
    tsim=False,  # tsim simulation might take several minutes to complete for the mobilenet
)
tool.convert_model()

## 10. Display inference timing

Summarize the selected target's `results.json` timing section, including estimated cycles, instructions, average CPI, latency, and throughput. Profile shares describe operator contributions to execution cost and are separate from classification quality.

In [ ]:
cs.display_inference_benchmark(model_folder, target)

In [ ]:
cs.plot_profile_shares(model_folder, target)

## 11. Compare performance across hardware platforms

Run the comparison target for the same exported model and representative data, then display conversion and execution results across AURIX™ and TRAVEO™ platforms.

In [ ]:
target = "compare"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    tsim=False,
)
tool.convert_model()

In [ ]:
df = cs.display_platform_comparison(model_folder)

## 12. References and Generated Artifacts

- **Dataset:** [Raindrop Clarity](https://github.com/jinyeying/RaindropClarity), Jin et al. (2024), a dual-focused dataset for day and night raindrop removal, repurposed here for clear-versus-raindrop classification.
- **Model reference:** [Searching for MobileNetV3](https://openaccess.thecvf.com/content_ICCV_2019/html/Howard_Searching_for_MobileNetV3_ICCV_2019_paper.html), Howard et al. (2019), and ImageNet1K pretrained weights [3].
- **Model artifacts:** `mobilenet_rain_drop` ONNX model, representative transformed image input, and expected output data.
- **Deployment artifacts:** generated source, binaries, logs, timing/profile results, and platform comparison output under `out/mobilenet_rain_drop/test_mobilenet_rain_drop/<target>/`.
- **Deployment targets:** AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.
- **Limitations:** the source dataset was designed for raindrop removal rather than classification, and the example uses a short transfer-learning run. Validate class balance, camera variation, weather coverage, and licensing before deployment.